# DeepPEF: Protein Stability Prediction (ddG)

**M.Sc. Thesis - Nissim Brami**

Full pipeline on Colab Pro GPU: SaProt embeddings + GNN training + ensemble.

**Runtime:** ~6-8 hours on T4/A100

## 0. GPU Check

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")
else:
    raise RuntimeError("No GPU\! Runtime > Change runtime type > GPU")

## 1. Clone & Install

In [ ]:
import os
REPO = "https://github.com/nissimbrami/DeepEF-Thesis.git"
WORK = "/content/DeepPEF"
if not os.path.exists(WORK):
    \!git clone {REPO} {WORK}
os.chdir(WORK)
\!pwd

In [ ]:
\!pip install -q torch-geometric transformers fair-esm wandb scipy scikit-learn tqdm pandas
import torch
tv = torch.__version__.split("+")[0]
cv = torch.version.cuda.replace(".","") if torch.version.cuda else "cpu"
\!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{tv}+cu{cv}.html
os.environ["WANDB_MODE"] = "disabled"

## 2. Mount Drive & Load Data

Upload  to Drive containing  etc.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DATA = "/content/drive/MyDrive/DeepPEF/DeepPEF_data.zip"
if os.path.exists(DRIVE_DATA):
    \!unzip -qo {DRIVE_DATA} -d {WORK}/
    print("Data extracted\!")
else:
    print(f"Upload data to {DRIVE_DATA}")

In [ ]:
# Verify
td = "./data/MsDs/training_data"
ps = sorted(os.listdir(td)) if os.path.exists(td) else []
print(f"Proteins: {len(ps)}")
if ps:
    for f in ["coords.pt","deltaG.pt","mask.pt","emb.pt"]:
        print(f"  {ps[0]}/{f}: {os.path.exists(os.path.join(td,ps[0],f))}")

## 3. Generate SaProt Embeddings (~2-4h on T4)

In [ ]:
fst = "./data/foldseek_3di/3di_tokens.tsv"
if os.path.exists(fst):
    \!wc -l {fst}
else:
    \!wget -q https://zenodo.org/records/7992926/files/AlphaFold_model_PDBs.zip -O AF.zip
    \!unzip -qo AF.zip -d AlphaFold_model_PDBs/
    \!wget -q https://mmseqs.com/foldseek/foldseek-linux-x86_64.tar.gz && tar xzf foldseek-linux-x86_64.tar.gz
    os.makedirs("data/foldseek_3di", exist_ok=True)
    \!./foldseek/bin/foldseek structureto3didescriptor AlphaFold_model_PDBs/ data/foldseek_3di/3di_tokens.tsv /tmp/fs

In [ ]:
\!python data_creation/generate_saprot_embeddings.py --force

In [ ]:
# Validate
import torch
td = "./data/MsDs/training_data"
for p in sorted(os.listdir(td))[:5]:
    sp = os.path.join(td,p,"saprot_wt.pt")
    if os.path.exists(sp):
        e=torch.load(sp,weights_only=True)
        c=torch.load(os.path.join(td,p,"coords.pt"),weights_only=False)
        print(f"  {p}: {e.shape} (expect [{c.shape[0]},1280])")

## 4. Stage 1: ProtT5 Baseline (PCC ~0.50-0.52)

In [ ]:
os.makedirs("logs",exist_ok=True)
os.makedirs("Megascale-fineTuning/models",exist_ok=True)
\!python Megascale-fineTuning/pnas_train.py \n  --model_name stage1_prott5_seed42 --seed 42 --dataset_type pnas \n  --epochs 15 --epochs_freeze 15 --epochs_unfreeze 0 --no_pretrained \n  --loss_type huber_rank --ranking_weight 0.1 --use_knn_gat \n  --one_mut --dg_ml --cosine_lr --lr_min 1e-6 --weight_decay 1e-5 \n  2>&1 | tee logs/stage1_prott5.log

In [ ]:
\!grep -i "pearson\|pcc\|corr" logs/stage1_prott5.log | tail -5

## 5. Stage 2: SaProt (PCC 0.55-0.60)

In [ ]:
\!python Megascale-fineTuning/pnas_train.py \n  --model_name stage2_saprot_seed42 --seed 42 --dataset_type pnas \n  --epochs 15 --epochs_freeze 15 --epochs_unfreeze 0 --no_pretrained \n  --loss_type huber_rank --ranking_weight 0.1 --use_knn_gat \n  --one_mut --dg_ml --cosine_lr --lr_min 1e-6 --weight_decay 1e-5 \n  --emb_type saprot \n  2>&1 | tee logs/stage2_saprot.log

In [ ]:
\!grep -i "pearson\|pcc\|corr" logs/stage2_saprot.log | tail -5

## 6. Ensemble (5 Seeds) - +0.02-0.04 PCC

In [ ]:
BEST_EMB = "saprot"  # Change if Stage 1 > Stage 2
for seed in [42, 123, 456, 789, 1337]:
    print(f"
===== Seed {seed} =====")
    \!python Megascale-fineTuning/pnas_train.py \n      --model_name ensemble_{BEST_EMB}_seed{seed} --seed {seed} --dataset_type pnas \n      --epochs 15 --epochs_freeze 15 --epochs_unfreeze 0 --no_pretrained \n      --loss_type huber_rank --ranking_weight 0.1 --use_knn_gat \n      --one_mut --dg_ml --cosine_lr --lr_min 1e-6 --weight_decay 1e-5 \n      --emb_type {BEST_EMB} \n      2>&1 | tee logs/ensemble_{BEST_EMB}_seed{seed}.log

In [ ]:
print("RESULTS:")
for seed in [42,123,456,789,1337]:
    log=f"logs/ensemble_{BEST_EMB}_seed{seed}.log"
    if os.path.exists(log):
        print(f"Seed {seed}:")
        \!grep -i "pearson\|pcc" {log} | tail -1

## 7. Save to Drive

In [ ]:
SAVE="/content/drive/MyDrive/DeepPEF/results"
os.makedirs(SAVE,exist_ok=True)
\!cp -r logs/ {SAVE}/
\!cp -r Megascale-fineTuning/models/ {SAVE}/
print(f"Saved to {SAVE}")

## Summary
| Stage | Embedding | PCC |
|-------|-----------|-----|
| 1 | ProtT5 | ___ |
| 2 | SaProt | ___ |
| 4 | Ensemble | ___ |

Target: PCC >= 0.70